# Project 4: Financial Market Analysis
## Stock Portfolio Analysis & Risk Assessment

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from utils.data_analysis_utils import utils
print("✅ Imports OK")


In [ ]:
np.random.seed(42)
stocks = {
    'AAPL':{'sector':'Technology','beta':1.2,'vol':0.25},
    'GOOGL':{'sector':'Technology','beta':1.1,'vol':0.23},
    'MSFT':{'sector':'Technology','beta':1.0,'vol':0.22},
    'AMZN':{'sector':'Consumer',  'beta':1.3,'vol':0.28},
    'TSLA':{'sector':'Automotive','beta':1.8,'vol':0.45},
    'JPM': {'sector':'Financial', 'beta':1.1,'vol':0.24},
    'JNJ': {'sector':'Healthcare','beta':0.7,'vol':0.15},
    'PG':  {'sector':'Consumer',  'beta':0.6,'vol':0.14},
    'XOM': {'sector':'Energy',    'beta':1.0,'vol':0.22},
    'V':   {'sector':'Financial', 'beta':1.0,'vol':0.20},
}
n_days = 750
dates  = pd.date_range('2022-01-01', periods=n_days, freq='D')
market = np.random.normal(0.0005, 0.015, n_days)

ret_dict, px_dict = {}, {}
for s, info in stocks.items():
    alpha = np.random.normal(0.0002, 0.001)
    rets  = alpha + info['beta']*market + np.random.normal(0, info['vol']*0.5, n_days)
    ret_dict[s] = rets
    px_dict[s]  = 100 * np.exp(np.cumsum(rets))

returns_df = pd.DataFrame(ret_dict, index=dates)
prices_df  = pd.DataFrame(px_dict,  index=dates)
print(f"✅ {len(stocks)} stocks  |  {n_days} trading days")


In [ ]:
os.makedirs('visualizations', exist_ok=True)

perf = pd.DataFrame({
    'Total_Return': (prices_df.iloc[-1]/prices_df.iloc[0]-1)*100,
    'Annual_Return': ((prices_df.iloc[-1]/prices_df.iloc[0])**(252/n_days)-1)*100,
    'Volatility':   returns_df.std()*np.sqrt(252)*100,
    'Sharpe':       (returns_df.mean()*252)/(returns_df.std()*np.sqrt(252)),
}).round(2)

fig, axes = plt.subplots(2,2,figsize=(15,10))
for s in stocks: axes[0,0].plot(prices_df[s], label=s, alpha=0.7)
axes[0,0].set_title('Stock Price Trends', fontweight='bold'); axes[0,0].legend(fontsize=8)
axes[0,0].grid(True,alpha=0.3)

perf.sort_values('Total_Return').plot.barh(y='Total_Return', ax=axes[0,1], legend=False)
axes[0,1].set_title('Total Returns (%)', fontweight='bold')

axes[1,0].scatter(perf['Volatility'],perf['Annual_Return'],s=100)
for s,r in perf.iterrows(): axes[1,0].annotate(s,(r['Volatility'],r['Annual_Return']))
axes[1,0].set_xlabel('Volatility (%)'); axes[1,0].set_ylabel('Annual Return (%)')
axes[1,0].set_title('Risk-Return Tradeoff', fontweight='bold'); axes[1,0].grid(True,alpha=0.3)

perf.sort_values('Sharpe').plot.barh(y='Sharpe', ax=axes[1,1], color='teal', legend=False)
axes[1,1].set_title('Sharpe Ratio', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/stock_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: stock_performance.png")
print(perf)


In [ ]:
# Portfolio optimisation – efficient frontier
exp_ret = returns_df.mean()*252
cov     = returns_df.cov()*252
n_pf    = 5000
res     = np.zeros((3,n_pf))
wt_all  = []
for i in range(n_pf):
    w = np.random.dirichlet(np.ones(len(stocks)))
    wt_all.append(w)
    r = w @ exp_ret.values
    s = np.sqrt(w @ cov.values @ w)
    res[0,i]=r; res[1,i]=s; res[2,i]=r/s

best_i = np.argmax(res[2])
best_w = wt_all[best_i]

fig, ax = plt.subplots(figsize=(10,7))
sc = ax.scatter(res[1],res[0],c=res[2],cmap='viridis',alpha=0.5,s=8)
ax.scatter(res[1,best_i],res[0,best_i],c='red',marker='*',s=400,label='Max Sharpe')
ax.set_xlabel('Risk (Std Dev)'); ax.set_ylabel('Return')
ax.set_title('Efficient Frontier', fontweight='bold'); ax.legend()
plt.colorbar(sc,ax=ax,label='Sharpe')
plt.tight_layout()
plt.savefig('visualizations/efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Max Sharpe Portfolio  Return={res[0,best_i]*100:.2f}%  Risk={res[1,best_i]*100:.2f}%  Sharpe={res[2,best_i]:.3f}")


In [ ]:
# VaR analysis
pf_rets = returns_df.values @ best_w
var95 = np.percentile(pf_rets,(1-0.95)*100)
var99 = np.percentile(pf_rets,(1-0.99)*100)
cum   = (1+pd.Series(pf_rets,index=dates)).cumprod()
dd    = (cum - cum.expanding().max())/cum.expanding().max()*100

fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].hist(pf_rets,bins=60,alpha=0.7)
axes[0].axvline(var95,color='red',  linestyle='--',label=f'95% VaR {var95*100:.2f}%')
axes[0].axvline(var99,color='darkred',linestyle='--',label=f'99% VaR {var99*100:.2f}%')
axes[0].set_title('Returns Distribution with VaR', fontweight='bold'); axes[0].legend()

axes[1].fill_between(dd.index,dd,0,alpha=0.4,color='coral')
axes[1].plot(dd.index,dd,color='coral')
axes[1].axhline(dd.min(),color='red',linestyle='--',label=f'Max DD {dd.min():.1f}%')
axes[1].set_title('Portfolio Drawdown', fontweight='bold'); axes[1].legend()
plt.tight_layout()
plt.savefig('visualizations/risk_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"95% VaR={var95*100:.2f}%  99% VaR={var99*100:.2f}%  Max Drawdown={dd.min():.2f}%")


In [ ]:
utils.plot_correlation_matrix(returns_df, save_path='visualizations/correlation_matrix.png')
returns_df.to_csv('stock_returns.csv')
prices_df.to_csv('stock_prices.csv')
perf.to_csv('stock_performance.csv')
print("✅ Project 4 complete.")
